# Peptide Hit Prioritization — Google Colab (T4 GPU)

**Setup steps:**
1. Runtime → Change runtime type → Select **T4 GPU**
2. Run Cells 1–6 in order (proceed to Cell 7 after Cell 6 completes)
3. **Keep Cell 7 running** (prevents idle disconnection + displays URL)

> Caching the Boltz model weights (~5 GB) to Google Drive speeds up subsequent runs.

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('❌ No GPU detected. Please change the runtime type to T4 GPU.')
    raise SystemExit('GPU required')

In [ ]:
# ── Cell 2: Mount Google Drive (for model weight caching) ────────────────────
USE_DRIVE_CACHE = True  # Set to False to re-download every session

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    BOLTZ_CACHE_DIR = '/content/drive/MyDrive/boltz_cache'
    import os
    os.makedirs(BOLTZ_CACHE_DIR, exist_ok=True)
    if not os.path.exists(os.path.expanduser('~/.boltz')):
        os.symlink(BOLTZ_CACHE_DIR, os.path.expanduser('~/.boltz'))
    print('✅ Drive cache configured:', BOLTZ_CACHE_DIR)
else:
    print('ℹ️  No Drive cache (model will be re-downloaded each session)')

In [ ]:
# ── Cell 3: Clone repository ──────────────────────────────────────────────────
import getpass, os

GITHUB_USER  = 'rkikuchi-bis'
GITHUB_REPO  = 'peptide_v2'
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token: ')
repo_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

if os.path.exists(f'/content/{GITHUB_REPO}'):
    print('Updating existing directory...')
    !cd /content/{GITHUB_REPO} && git pull
else:
    !git clone {repo_url} /content/{GITHUB_REPO}

os.chdir(f'/content/{GITHUB_REPO}')
print('✅ Working directory:', os.getcwd())

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

packages = [
    'biopython==1.84',
    'streamlit>=1.36.0',
    'py3dmol>=2.0.0',
    'gemmi>=0.6',
    'altair>=5.0',
    'huggingface-hub>=0.20',
    'einops>=0.7',
    'biotite>=0.38',
    'boltz==2.2.1',
    'prodigy-prot',
]

print('Installing packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Installation failed')
print('✅ Installation complete')

In [ ]:
# ── Cell 5: Pre-download Boltz model weights ──────────────────────────────────
import os, subprocess, tempfile, yaml
from pathlib import Path

ckpt_path = Path(os.path.expanduser('~/.boltz')) / 'boltz1_conf.ckpt'

if ckpt_path.exists():
    print(f'✅ Already cached ({ckpt_path.stat().st_size/1e9:.1f} GB) — skipping')
else:
    print('Downloading model weights (first time only, ~5–10 min)...')
    dummy_yaml = {'version': 1, 'sequences': [{'protein': {'id': 'A', 'sequence': 'ACDEFGHIKLMNPQRSTVWY'}}]}
    with tempfile.TemporaryDirectory() as tmpdir:
        yaml_path = Path(tmpdir) / 'dummy.yaml'
        yaml_path.write_text(yaml.dump(dummy_yaml))
        subprocess.run(['boltz', 'predict', str(yaml_path), '--out_dir', tmpdir,
                        '--model', 'boltz1', '--accelerator', 'gpu',
                        '--sampling_steps', '1', '--use_msa_server'],
                       capture_output=True)
    print('✅ Download complete' if ckpt_path.exists() else '⚠️  Failed. Will be auto-downloaded in Cell 6.')

In [ ]:
# ── Cell 6: Launch Streamlit + cloudflared (completes immediately) ────────────
# This cell completes immediately. Processes continue running in the background.

import subprocess, time, re, sys, os
from pathlib import Path

PORT = 8501
CLOUDFLARED = Path('/usr/local/bin/cloudflared')

# Download cloudflared binary
if not CLOUDFLARED.exists():
    print('Downloading cloudflared...')
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(CLOUDFLARED)], check=True)
    CLOUDFLARED.chmod(0o755)

# Stop existing processes before restarting
os.system('pkill -f "streamlit run" 2>/dev/null; pkill -f cloudflared 2>/dev/null')
time.sleep(2)

# Launch Streamlit in background
with open('/tmp/streamlit.log', 'w') as log:
    st_proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.port', str(PORT),
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=log, stderr=log
    )
Path('/tmp/st_pid').write_text(str(st_proc.pid))
time.sleep(5)

# Launch cloudflared tunnel
# Write stderr to file (subprocess.PIPE buffer fills up in minutes and causes crashes)
Path('/tmp/cloudflared.log').write_text('')
with open('/tmp/cloudflared.log', 'w') as cf_log:
    cf_proc = subprocess.Popen(
        [str(CLOUDFLARED), 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.DEVNULL,
        stderr=cf_log
    )
Path('/tmp/cf_pid').write_text(str(cf_proc.pid))

# Retrieve URL: poll file for up to 30 seconds
url = None
for _ in range(30):
    time.sleep(1)
    try:
        text = Path('/tmp/cloudflared.log').read_text()
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', text)
        if m:
            url = m.group(0)
            break
    except Exception:
        pass

Path('/tmp/app_url').write_text(url or '')

print('='*60)
print('✅ App launched!')
print(f'   URL: {url or "Failed to retrieve — re-run Cell 6"}')
print('='*60)
print('Now run Cell 7 to prevent idle disconnection.')


In [ ]:
# ── Cell 7: Keep-alive + monitoring (keep this cell running) ──────────────────
# Cell 6 processes are running in the background.
# This cell prevents Colab's idle timeout (90 min).
# If cloudflared or Streamlit goes down, it auto-restarts and displays the new URL.

import subprocess, time, re, sys, os
from pathlib import Path
from IPython.display import display, Javascript

PORT = 8501
CLOUDFLARED = Path('/usr/local/bin/cloudflared')

# Prevent Colab idle timeout (click the connect button every 55 seconds)
display(Javascript("""
  if (typeof keepAliveInterval === 'undefined') {
    window.keepAliveInterval = setInterval(function() {
      var btn = document.querySelector("#connect");
      if (btn) btn.click();
    }, 55000);
  }
"""))

def is_alive(pid_file):
    try:
        pid = int(Path(pid_file).read_text())
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def restart_streamlit():
    with open('/tmp/streamlit.log', 'a') as log:
        proc = subprocess.Popen(
            [sys.executable, '-m', 'streamlit', 'run', 'app.py',
             '--server.port', str(PORT), '--server.headless', 'true',
             '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
            stdout=log, stderr=log
        )
    Path('/tmp/st_pid').write_text(str(proc.pid))

def restart_cloudflared():
    # Write stderr to file (no PIPE)
    Path('/tmp/cloudflared.log').write_text('')
    with open('/tmp/cloudflared.log', 'w') as cf_log:
        proc = subprocess.Popen(
            [str(CLOUDFLARED), 'tunnel', '--url', f'http://localhost:{PORT}'],
            stdout=subprocess.DEVNULL, stderr=cf_log
        )
    Path('/tmp/cf_pid').write_text(str(proc.pid))
    # Poll for URL
    for _ in range(30):
        time.sleep(1)
        try:
            text = Path('/tmp/cloudflared.log').read_text()
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', text)
            if m:
                url = m.group(0)
                Path('/tmp/app_url').write_text(url)
                return url
        except Exception:
            pass
    return None

current_url = Path('/tmp/app_url').read_text() if Path('/tmp/app_url').exists() else 'unknown'
print(f'Monitoring started | URL: {current_url}')
print('Keep this cell running. Press ■ to stop.\n')

try:
    tick = 0
    while True:
        time.sleep(30)
        tick += 1

        # Monitor Streamlit
        if not is_alive('/tmp/st_pid'):
            print(f'[{tick}] ⚠️  Streamlit stopped → restarting...')
            restart_streamlit()
            time.sleep(5)
            print(f'[{tick}] ✅ Streamlit restarted')

        # Monitor cloudflared
        if not is_alive('/tmp/cf_pid'):
            print(f'[{tick}] ⚠️  cloudflared stopped → restarting...')
            new_url = restart_cloudflared()
            print(f'[{tick}] ✅ New URL: {new_url or "failed to retrieve"}')

        # Re-display URL every hour
        if tick % 120 == 0:
            url = Path('/tmp/app_url').read_text()
            print(f'[Running {tick*30//3600}h {(tick*30%3600)//60}m] URL: {url}')

except KeyboardInterrupt:
    print('\nMonitoring stopped. Processes are still running in the background.')
    print('To stop completely: !pkill -f streamlit; !pkill -f cloudflared')


## Troubleshooting

| Symptom | Solution |
|---------|----------|
| `GPU required` error | Runtime → Change runtime type → T4 GPU |
| URL not displayed | Re-run Cell 6 |
| Cell 7 stopped | Re-run Cell 7 (background app is still running) |
| Cannot connect to app | Re-run Cell 6 (a new URL will be issued) |
| Colab session expired (90 min) | Re-run Cell 6 → Cell 7 |
| Check logs | `!tail -50 /tmp/streamlit.log` |